In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Helper to convert min/max timestamps to UTC+8 formatted strings
def fmt_min_max_ts(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    dt = pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz)
    return dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Load files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Summarize metric.csv
metric_rows = metric_df.shape[0]
metric_min_ts = fmt_min_max_ts(metric_df['timestamp'].min()) if 'timestamp' in metric_df.columns else None
metric_max_ts = fmt_min_max_ts(metric_df['timestamp'].max()) if 'timestamp' in metric_df.columns else None
metric_unique_cmdb = list(pd.Index(metric_df['cmdb_id'].dropna().unique()[:50]))
metric_unique_kpis = list(pd.Index(metric_df['kpi_name'].dropna().unique()[:50]))
metric_sample = metric_df.head(5).copy()
if 'timestamp' in metric_sample.columns:
    metric_sample['ts_utc8'] = pd.to_datetime(metric_sample['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_sample = metric_sample[['ts_utc8','cmdb_id','kpi_name','value']]

metric_summary = pd.DataFrame([{
    'file':'metric.csv',
    'rows': metric_rows,
    'min_ts': metric_min_ts,
    'max_ts': metric_max_ts,
    'unique_cmdb_count': int(metric_df['cmdb_id'].nunique(dropna=True)),
    'unique_cmdb_sample': metric_unique_cmdb,
    'unique_kpi_sample': metric_unique_kpis
}])

# Summarize trace.csv
trace_rows = trace_df.shape[0]
trace_min_ts = fmt_min_max_ts(trace_df['timestamp'].min()) if 'timestamp' in trace_df.columns else None
trace_max_ts = fmt_min_max_ts(trace_df['timestamp'].max()) if 'timestamp' in trace_df.columns else None
trace_unique_cmdb = list(pd.Index(trace_df['cmdb_id'].dropna().unique()[:50]))
trace_unique_names = list(pd.Index(trace_df['trace_name'].dropna().unique()[:50]))
trace_sample = trace_df.head(5).copy()
if 'timestamp' in trace_sample.columns:
    trace_sample['ts_utc8'] = pd.to_datetime(trace_sample['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    trace_sample = trace_sample[['ts_utc8','cmdb_id','trace_name','value']]

trace_summary = pd.DataFrame([{
    'file':'trace.csv',
    'rows': trace_rows,
    'min_ts': trace_min_ts,
    'max_ts': trace_max_ts,
    'unique_cmdb_count': int(trace_df['cmdb_id'].nunique(dropna=True)),
    'unique_trace_name_sample': trace_unique_names,
    'unique_cmdb_sample': trace_unique_cmdb
}])

# Summarize log.csv
log_rows = log_df.shape[0]
log_min_ts = fmt_min_max_ts(log_df['timestamp'].min()) if 'timestamp' in log_df.columns else None
log_max_ts = fmt_min_max_ts(log_df['timestamp'].max()) if 'timestamp' in log_df.columns else None
log_unique_cmdb = list(pd.Index(log_df['cmdb_id'].dropna().unique()[:50]))
log_unique_names = list(pd.Index(log_df['log_name'].dropna().unique()[:50]))
log_sample = log_df.head(5).copy()
if 'timestamp' in log_sample.columns:
    log_sample['ts_utc8'] = pd.to_datetime(log_sample['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    log_sample = log_sample[['ts_utc8','cmdb_id','log_name','value']]

log_summary = pd.DataFrame([{
    'file':'log.csv',
    'rows': log_rows,
    'min_ts': log_min_ts,
    'max_ts': log_max_ts,
    'unique_cmdb_count': int(log_df['cmdb_id'].nunique(dropna=True)),
    'unique_log_name_sample': log_unique_names,
    'unique_cmdb_sample': log_unique_cmdb
}])

# Summarize error_logs.csv
error_rows = error_df.shape[0]
error_min_ts = fmt_min_max_ts(error_df['timestamp'].min()) if 'timestamp' in error_df.columns else None
error_max_ts = fmt_min_max_ts(error_df['timestamp'].max()) if 'timestamp' in error_df.columns else None
error_unique_cmdb = list(pd.Index(error_df['cmdb_id'].dropna().unique()[:50]))
# Prepare message snippets (full if short, else truncated)
_unique_msgs = list(pd.Index(error_df['message'].dropna().unique()))
def snippet(m, n=120):
    if m is None:
        return m
    m = str(m)
    return m if len(m) <= n else (m[:n] + '...')
error_unique_msg_snippets = [snippet(m, 120) for m in _unique_msgs[:50]]

error_sample = error_df.head(5).copy()
if 'timestamp' in error_sample.columns:
    error_sample['ts_utc8'] = pd.to_datetime(error_sample['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
# truncate message column for compactness
if 'message' in error_sample.columns:
    error_sample['message_snippet'] = error_sample['message'].astype(str).apply(lambda x: x if len(x) <= 200 else x[:200] + '...')
    error_sample = error_sample[['ts_utc8','cmdb_id','message_snippet']]

error_summary = pd.DataFrame([{
    'file':'error_logs.csv',
    'rows': error_rows,
    'min_ts': error_min_ts,
    'max_ts': error_max_ts,
    'unique_cmdb_count': int(error_df['cmdb_id'].nunique(dropna=True)),
    'unique_message_snippets': error_unique_msg_snippets,
    'unique_cmdb_sample': error_unique_cmdb
}])

# Display compact summaries and samples (variables)
metric_summary, metric_sample, trace_summary, trace_sample, log_summary, log_sample, error_summary, error_sample

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry files:\n"
    "- metric.csv: 321,454 rows; timestamps from 2022-03-21 01:30 to 2022-03-21 02:30 (UTC+8); 59 unique cmdb_id values. "
    "Example KPIs include app.grpc.count, app.grpc.mrt, app.http.count. Sample rows show adservice metrics at 2022-03-21 01:30 (various KPI values).\n"
    "- trace.csv: 29,280 rows; timestamps from 2022-03-21 02:00 to 2022-03-21 02:29 (UTC+8); 40 unique cmdb_id values. "
    "Example trace names include trace.from_frontend-0.duration_mean and related latency/error metrics. Sample rows show adservice-0 traces at 2022-03-21 02:00.\n"
    "- log.csv: 1,728 rows; timestamps from 2022-03-21 02:00 to 2022-03-21 02:29 (UTC+8); 31 unique cmdb_id values. "
    "Log keys seen: log.error_count and log.row_count. Samples show zero error_count and per-minute row_count values for adservice pods.\n"
    "- error_logs.csv: empty (0 rows); no timestamps, cmdb_id, or messages present.\n"
    "All timestamps reported above are in UTC+8."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file    rows                   min_ts                   max_ts  unique_cmdb_count                                 unique_cmdb_sample                                  unique_kpi_sample
0  metric.csv  321454  2022-03-21 01:30:00 CST  2022-03-21 02:30:00 CST                 59  [adservice, adservice-0, adservice-1, adservic...  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...,                     ts_utc8    cmdb_id        kpi_name       value
0 2022-03-21 01:30:00+08:00  adservice  app.grpc.count   77.000000
1 2022-03-21 01:30:00+08:00  adservice    app.grpc.mrt    0.002466
2 2022-03-21 01:30:00+08:00  adservice     app.grpc.rr  100.000000
3 2022-03-21 01:30:00+08:00  adservice     app.grpc.sr  100.000000
4 2022-03-21 01:30:00+08:00  adservice  app.http.count   24.000000,         file   rows                   min_ts                   max_ts  unique_cmdb_count                           unique_trace_name_sample                                 unique_cmdb_sample
0  trace.csv  29280  2022-03-21 02:00:00 CST  2022-03-21 02:29:00 CST                 40  [trace.from_frontend-0.duration_mean, trace.fr...  [adservice-0, adservice-1, adservice-2, adserv...,                     ts_utc8      cmdb_id                           trace_name     value
0 2022-03-21 02:00:00+08:00  adservice-0  trace.from_frontend-0.duration_mean  0.000025
1 2022-03-21 02:00:00+08:00  adservice-0   trace.from_frontend-0.duration_p95  0.000054
2 2022-03-21 02:00:00+08:00  adservice-0     trace.from_frontend-0.error_rate  0.000000
3 2022-03-21 02:00:00+08:00  adservice-0      trace.from_frontend-0.row_count  9.000000
4 2022-03-21 02:00:00+08:00  adservice-0  trace.from_frontend-1.duration_mean  0.000025,       file  rows                   min_ts                   max_ts  unique_cmdb_count            unique_log_name_sample                                 unique_cmdb_sample
0  log.csv  1728  2022-03-21 02:00:00 CST  2022-03-21 02:29:00 CST                 31  [log.error_count, log.row_count]  [adservice-0, adservice-1, adservice-2, cartse...,                     ts_utc8      cmdb_id         log_name  value
0 2022-03-21 02:00:00+08:00  adservice-0  log.error_count    0.0
1 2022-03-21 02:00:00+08:00  adservice-0    log.row_count   54.0
2 2022-03-21 02:00:00+08:00  adservice-1  log.error_count    0.0
3 2022-03-21 02:00:00+08:00  adservice-1    log.row_count   40.0
4 2022-03-21 02:00:00+08:00  adservice-2  log.error_count    0.0,              file  rows min_ts max_ts  unique_cmdb_count unique_message_snippets unique_cmdb_sample
0  error_logs.csv     0   None   None                  0                      []                 [], Empty DataFrame
Columns: [ts_utc8, cmdb_id, message_snippet]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# helper to format epoch seconds to UTC+8 string
def fmt_min_max_ts(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    dt = pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz)
    return dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Assume metric_df already loaded in the environment
# 1) Compute global P95 and P5 for each (cmdb_id, kpi_name)
thresholds = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P5=lambda x: x.quantile(0.05))
)

total_pairs = thresholds.shape[0]  # total unique (cmdb_id, kpi_name) pairs scanned

# 2) Filter metric.csv to the given UTC+8 time window (inclusive)
start_ts = int(pd.Timestamp('2022-03-21 02:00:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2022-03-21 02:30:00', tz=tz).timestamp())

window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) For each pair in the window calculate required stats using global thresholds
# Merge thresholds into windowed points so we can mark anomalies by global thresholds
merged = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# mark anomalies: value >= global_P95 OR value <= global_P5
merged['is_anom'] = ((merged['value'] >= merged['global_P95']) | (merged['value'] <= merged['global_P5'])).astype(int)

# group and compute stats per (cmdb_id,kpi_name) present in the window
def grp_stats(g):
    pts = int(g.shape[0])
    an = int(g['is_anom'].sum())
    ear = int(g.loc[g['is_anom']==1, 'timestamp'].min()) if an>0 else None
    mx = float(g['value'].max())
    mn = float(g['value'].min())
    return pd.Series({
        'points_in_window': pts,
        'anomaly_count': an,
        'earliest_anomaly_ts': ear,
        'max_value_in_window': mx,
        'min_value_in_window': mn
    })

window_stats = merged.groupby(['cmdb_id','kpi_name'], as_index=False).apply(grp_stats).reset_index()

# attach global thresholds
window_stats = window_stats.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# compute tie-break severity = max(|max - P95|, |min - P5|)
window_stats['severity'] = window_stats.apply(
    lambda r: max(abs(r['max_value_in_window'] - r['global_P95']), abs(r['min_value_in_window'] - r['global_P5'])), axis=1
)

# format earliest anomaly timestamp to UTC+8 string
window_stats['earliest_anomaly_ts'] = window_stats['earliest_anomaly_ts'].apply(lambda x: fmt_min_max_ts(x) if pd.notna(x) else None)

# 4) Top 20 table sorted by anomaly_count desc, then severity desc
top20 = (
    window_stats
    .sort_values(['anomaly_count','severity'], ascending=[False, False])
    .loc[:, ['cmdb_id','kpi_name','global_P95','global_P5','points_in_window','anomaly_count','earliest_anomaly_ts','max_value_in_window','min_value_in_window']]
    .head(20)
    .reset_index(drop=True)
)

# 5) total number of unique (cmdb_id, kpi_name) pairs scanned (global)
top20, total_pairs

```
Out[2]:
```
Summary (window 2022-03-21 02:00:00 — 2022-03-21 02:30:00 UTC+8):

- Total unique (cmdb_id, kpi_name) pairs scanned (global): 5,282.
- Top 20 pairs by anomaly count (each column shown in result): all top 20 have anomaly_count = 31 of 31 points in the window (i.e., every sample in the 30‑minute window met the anomaly rule: value >= global_P95 or value <= global_P5).
- Earliest anomaly for these top results is 2022-03-21 02:00:00 (UTC+8) for all listed pairs.
- The top issues are dominated by container/node memory metrics (e.g., container_memory_rss, container_memory_mapped_file, runtime/java memory/GarbageCollector info, container_memory_failures) mostly on node-6 (many recommendationservice/paymentservice/adservice related pods) and node-5 (shippingservice). Example entries:
  - recommendationservice-2 / container.node-6.container_memory_rss — global_P95 39,837,696; global_P5 39,718,912; max_in_window 48,285,696; min_in_window ~39,718,910; anomaly_count 31.
  - recommendationservice-2 / container.node-6.container_memory_mapped_file — global_P95 675,840; global_P5 405,504; max 3,244,032; anomaly_count 31.
  - paymentservice-2 / container.node-6.container_memory_rss — global_P95 34,316,288; global_P5 33,861,632; anomaly_count 31.
  - adservice-1 / container.node-6.container_memory_rss — global_P95 139,051,008; global_P5 138,915,840; anomaly_count 31.
  - Several container_memory_failures metrics show global_P95 = global_P5 = 0 but window max > 0 (i.e., failures observed throughout the window).
- Other top anomalies include a node IO metric (node-2 system.io.rkb_s) and filesystem read metrics showing constant detections across the window.
- Interpretation: during the 02:00–02:30 UTC+8 window many container memory-related KPIs (particularly on node-6 and some on node-5) were outside their global expected ranges for every sampled minute, indicating sustained memory-related deviations for those pods/services across the window.

If you want, I can (1) expand the top table with exact numeric formatting for all 20 rows, (2) provide per-service aggregation (grouping pods into services), or (3) inspect corresponding trace/log records for those affected cmdb_ids. Which would you prefer next?

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                           kpi_name   global_P95    global_P5  points_in_window  anomaly_count      earliest_anomaly_ts  max_value_in_window  min_value_in_window
0    recommendationservice-2              container.node-6.container_memory_rss   39837696.0   39718912.0              31.0           31.0  2022-03-21 02:00:00 CST           48285696.0         3.971891e+07
1    recommendationservice-2      container.node-6.container_memory_mapped_file     675840.0     405504.0              31.0           31.0  2022-03-21 02:00:00 CST            3244032.0         4.055040e+05
2           paymentservice-2              container.node-6.container_memory_rss   34316288.0   33861632.0              31.0           31.0  2022-03-21 02:00:00 CST           33861632.0         3.386163e+07
3                adservice-1              container.node-6.container_memory_rss  139051008.0  138915840.0              31.0           31.0  2022-03-21 02:00:00 CST          139051008.0         1.390510e+08
4          shippingservice-2              container.node-5.container_memory_rss   13078528.0   13062144.0              31.0           31.0  2022-03-21 02:00:00 CST           13078528.0         1.292698e+07
5                 adservice2  runtime.java_lang_OperatingSystem_FreePhysical...   62574592.0   62554112.0              31.0           31.0  2022-03-21 02:00:00 CST           62574592.0         6.243123e+07
6             emailservice-0      container.node-6.container_memory_mapped_file    1622016.0    1486848.0              31.0           31.0  2022-03-21 02:00:00 CST            1622016.0         1.419264e+06
7    recommendationservice-1  container.node-6.container_memory_failures.con...          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST              10098.0         0.000000e+00
8    recommendationservice-1  container.node-6.container_memory_failures.hie...          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST              10098.0         0.000000e+00
9    recommendationservice-2  container.node-6.container_memory_failures.con...          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST               9058.5         0.000000e+00
10   recommendationservice-2  container.node-6.container_memory_failures.hie...          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST               9058.5         0.000000e+00
11  recommendationservice2-0  container.node-6.container_memory_failures.con...          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST               5659.5         0.000000e+00
12  recommendationservice2-0  container.node-6.container_memory_failures.hie...          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST               5659.5         0.000000e+00
13   recommendationservice-0  container.node-6.container_memory_failures.con...          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST               4504.5         0.000000e+00
14   recommendationservice-0  container.node-6.container_memory_failures.hie...          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST               4504.5         0.000000e+00
15                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...   27656192.0   27656192.0              31.0           31.0  2022-03-21 02:00:00 CST           27656192.0         2.765392e+07
16         shippingservice-2  container.node-5.container_memory_failures.con...          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST                 82.5         0.000000e+00
17         shippingservice-2  container.node-5.container_memory_failures.hie...          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST                 82.5         0.000000e+00
18                    node-2                                    system.io.rkb_s          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST                 52.0         0.000000e+00
19  recommendationservice2-0       container.node-6.container_fs_reads./dev/vda          0.0          0.0              31.0           31.0  2022-03-21 02:00:00 CST                 38.5         0.000000e+00, 5282)```
```

In [ ]:
# In[3]:

import re

# Aggregate per cmdb_id from existing per-(cmdb_id,kpi_name) table `window_stats`
agg = (
    window_stats
    .groupby('cmdb_id', as_index=False)
    .agg(
        num_kpis_scanned=('kpi_name','nunique'),
        num_kpis_with_any_anomaly=('anomaly_count', lambda s: int((s>0).sum())),
        max_severity=('severity','max')
    )
)

# Compute earliest anomaly timestamp across all kpis per cmdb_id using `merged` (rows where is_anom==1)
earliest_anom = (
    merged[merged['is_anom']==1]
    .groupby('cmdb_id', as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_anom_ts_unix'})
)

# Format earliest timestamps to UTC+8 strings (reuse fmt_min_max_ts)
earliest_anom['earliest_anomaly_ts_across_kpis'] = earliest_anom['earliest_anom_ts_unix'].apply(lambda x: fmt_min_max_ts(x) if pd.notna(x) else None)
earliest_anom = earliest_anom[['cmdb_id','earliest_anomaly_ts_across_kpis']]

# Merge earliest anomaly info into agg
agg = agg.merge(earliest_anom, on='cmdb_id', how='left')

# Derive service_name: keep node-<n> as-is; else strip trailing -<number>
def derive_service(name):
    if re.fullmatch(r'node-\d+', name):
        return name
    return re.sub(r'-\d+$', '', name)

agg['service_name'] = agg['cmdb_id'].astype(str).apply(derive_service)

# Reorder columns as requested
agg = agg[['cmdb_id','service_name','num_kpis_scanned','num_kpis_with_any_anomaly','earliest_anomaly_ts_across_kpis','max_severity']]

# Top 20 sorted by num_kpis_with_any_anomaly desc, then max_severity desc
top20_cmdb = agg.sort_values(['num_kpis_with_any_anomaly','max_severity'], ascending=[False, False]).head(20).reset_index(drop=True)

# Total distinct cmdb_id scanned (in the metric-level aggregates table)
total_distinct_cmdb = int(agg['cmdb_id'].nunique())

top20_cmdb, total_distinct_cmdb

```
Out[3]:
```
Summary (per-cmdb_id aggregates, top 20 shown; window 2022-03-21 02:00–02:30 UTC+8):

Total distinct cmdb_id scanned: 59

Top 20 cmdb_id (sorted by number of kpis with anomalies, then severity):
1. adservice (adservice) — scanned 341 KPIs, 341 KPIs with anomalies, earliest anomaly 2022-03-21 02:00:00, max_severity ≈ 4.66e+09
2. adservice2 (adservice2) — 333 scanned, 333 anomalous, earliest 02:00:00, max_severity ≈ 4.79e+09
3. frontend2-0 (frontend2) — 196 scanned, 196 anomalous, earliest 02:00:00, max_severity ≈ 3.81e+04
4. frontend-0 (frontend) — 188 scanned, 188 anomalous, earliest 02:00:00, max_severity ≈ 3.67e+04
5. frontend-1 (frontend) — 188 scanned, 188 anomalous, earliest 02:00:00, max_severity ≈ 3.64e+04
6. frontend-2 (frontend) — 188 scanned, 188 anomalous, earliest 02:00:00, max_severity ≈ 9.03e+03
7. checkoutservice-1 (checkoutservice) — 144 scanned, 144 anomalous, earliest 02:00:00, max_severity ≈ 3.48e+03
8. checkoutservice2-0 (checkoutservice2) — 144 scanned, 144 anomalous, earliest 02:00:00, max_severity ≈ 1.78e+03
9. checkoutservice-2 (checkoutservice) — 140 scanned, 140 anomalous, earliest 02:00:00, max_severity ≈ 9.59e+05
10. checkoutservice-0 (checkoutservice) — 140 scanned, 140 anomalous, earliest 02:00:00, max_severity ≈ 3.48e+03
11. productcatalogservice-1 (productcatalogservice) — 120 scanned, 120 anomalous, earliest 02:00:00, max_severity ≈ 1.23e+04
12. productcatalogservice-0 (productcatalogservice) — 120 scanned, 120 anomalous, earliest 02:00:00, max_severity ≈ 6.18e+03
13. productcatalogservice-2 (productcatalogservice) — 116 scanned, 116 anomalous, earliest 02:00:00, max_severity ≈ 6.76e+05
14. productcatalogservice2-0 (productcatalogservice2) — 104 scanned, 104 anomalous, earliest 02:00:00, max_severity ≈ 1.73e+04
15. recommendationservice2-0 (recommendationservice2) — 102 scanned, 102 anomalous, earliest 02:00:00, max_severity ≈ 5.66e+03
16. cartservice-2 (cartservice) — 100 scanned, 100 anomalous, earliest 02:00:00, max_severity ≈ 1.96e+06
17. cartservice-0 (cartservice) — 100 scanned, 100 anomalous, earliest 02:00:00, max_severity ≈ 1.49e+06
18. cartservice2-0 (cartservice2) — 100 scanned, 99 anomalous, earliest 02:00:00, max_severity ≈ 1.86e+06
19. recommendationservice-2 (recommendationservice) — 98 scanned, 98 anomalous, earliest 02:00:00, max_severity ≈ 8.45e+06
20. recommendationservice-1 (recommendationservice) — 98 scanned, 98 anomalous, earliest 02:00:00, max_severity ≈ 1.01e+04

Key observations:
- Many cmdb_ids have all (or nearly all) scanned KPIs flagged as anomalous in the 02:00–02:30 window.
- Earliest anomalies for the top entries all occur at the window start (2022-03-21 02:00:00 UTC+8).
- Severity (max deviation) varies widely: some services show very large numeric deviations (e.g., adservice/adservice2), others show large but smaller deviations (memory/IO related for cart/product/recommendation services).

If you want, I can produce this same table as CSV-like rows or focus on a specific service/cmdb_id for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id            service_name  num_kpis_scanned  num_kpis_with_any_anomaly earliest_anomaly_ts_across_kpis  max_severity
0                  adservice               adservice               341                        341         2022-03-21 02:00:00 CST  4.661667e+09
1                 adservice2              adservice2               333                        333         2022-03-21 02:00:00 CST  4.785833e+09
2                frontend2-0               frontend2               196                        196         2022-03-21 02:00:00 CST  3.810000e+04
3                 frontend-0                frontend               188                        188         2022-03-21 02:00:00 CST  3.665000e+04
4                 frontend-1                frontend               188                        188         2022-03-21 02:00:00 CST  3.637500e+04
5                 frontend-2                frontend               188                        188         2022-03-21 02:00:00 CST  9.025000e+03
6          checkoutservice-1         checkoutservice               144                        144         2022-03-21 02:00:00 CST  3.475000e+03
7         checkoutservice2-0        checkoutservice2               144                        144         2022-03-21 02:00:00 CST  1.775000e+03
8          checkoutservice-2         checkoutservice               140                        140         2022-03-21 02:00:00 CST  9.584640e+05
9          checkoutservice-0         checkoutservice               140                        140         2022-03-21 02:00:00 CST  3.475000e+03
10   productcatalogservice-1   productcatalogservice               120                        120         2022-03-21 02:00:00 CST  1.232500e+04
11   productcatalogservice-0   productcatalogservice               120                        120         2022-03-21 02:00:00 CST  6.175000e+03
12   productcatalogservice-2   productcatalogservice               116                        116         2022-03-21 02:00:00 CST  6.758400e+05
13  productcatalogservice2-0  productcatalogservice2               104                        104         2022-03-21 02:00:00 CST  1.732500e+04
14  recommendationservice2-0  recommendationservice2               102                        102         2022-03-21 02:00:00 CST  5.659500e+03
15             cartservice-2             cartservice               100                        100         2022-03-21 02:00:00 CST  1.959936e+06
16             cartservice-0             cartservice               100                        100         2022-03-21 02:00:00 CST  1.486848e+06
17            cartservice2-0            cartservice2               100                         99         2022-03-21 02:00:00 CST  1.863680e+06
18   recommendationservice-2   recommendationservice                98                         98         2022-03-21 02:00:00 CST  8.448000e+06
19   recommendationservice-1   recommendationservice                98                         98         2022-03-21 02:00:00 CST  1.009800e+04, 59)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse timezone and helper fmt_min_max_ts from earlier session
# tz, fmt_min_max_ts, start_ts, end_ts, trace_df are expected to exist in the kernel

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using entire trace series
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P5=lambda x: x.quantile(0.05))
)

total_pairs_trace = int(trace_thresholds.shape[0])

# 2) Filter trace.csv to the UTC+8 window (inclusive)
window_trace = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds and mark anomalies using global thresholds computed above
merged_trace = window_trace.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# safe anomaly marker (handle possible NaNs in thresholds)
merged_trace['is_anom'] = merged_trace.apply(
    lambda r: 1 if (pd.notna(r['global_P95']) and pd.notna(r['global_P5']) and ((r['value'] >= r['global_P95']) or (r['value'] <= r['global_P5']))) else 0,
    axis=1
)

# Group to compute required stats per (cmdb_id, trace_name) present in the window
def trace_grp_stats(g):
    pts = int(g.shape[0])
    an = int(g['is_anom'].sum())
    ear = int(g.loc[g['is_anom']==1, 'timestamp'].min()) if an>0 else None
    mx = float(g['value'].max())
    mn = float(g['value'].min())
    return pd.Series({
        'points_in_window': pts,
        'anomaly_count': an,
        'earliest_anomaly_ts': ear,
        'max_value_in_window': mx,
        'min_value_in_window': mn
    })

if merged_trace.shape[0] > 0:
    trace_window_stats = merged_trace.groupby(['cmdb_id','trace_name'], as_index=False).apply(trace_grp_stats).reset_index()
    # Attach global thresholds
    trace_window_stats = trace_window_stats.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    # severity tie-breaker
    trace_window_stats['severity'] = trace_window_stats.apply(
        lambda r: max(abs(r['max_value_in_window'] - r['global_P95']), abs(r['min_value_in_window'] - r['global_P5'])),
        axis=1
    )
    # format earliest anomaly ts to UTC+8 strings
    trace_window_stats['earliest_anomaly_ts'] = trace_window_stats['earliest_anomaly_ts'].apply(lambda x: fmt_min_max_ts(x) if pd.notna(x) else None)

    # 4) Top 20 sorted by anomaly_count desc then severity desc
    top20_trace = (
        trace_window_stats
        .sort_values(['anomaly_count','severity'], ascending=[False, False])
        .loc[:, ['cmdb_id','trace_name','global_P95','global_P5','points_in_window','anomaly_count','earliest_anomaly_ts','max_value_in_window','min_value_in_window']]
        .head(20)
        .reset_index(drop=True)
    )
else:
    # Empty results if no points in window
    top20_trace = pd.DataFrame(columns=['cmdb_id','trace_name','global_P95','global_P5','points_in_window','anomaly_count','earliest_anomaly_ts','max_value_in_window','min_value_in_window'])

# Display compact outputs: top20_trace and total distinct pairs scanned
top20_trace, total_pairs_trace

```
Out[4]:
```
Summary (trace window 2022-03-21 02:00:00 — 2022-03-21 02:30:00 UTC+8):

- Total distinct (cmdb_id, trace_name) pairs scanned (global): 1,232.

- Top 20 pairs by anomaly_count (all shown rows are in the result):
  - Every top item has points_in_window = 30 and anomaly_count = 30 (i.e., every sampled minute in the window was flagged).
  - Earliest anomaly for these top pairs is 2022-03-21 02:00:00 (UTC+8).
  - Examples:
    - cartservice-0 / trace.self.duration_p95 — global_P95=0.0, global_P5=0.0, points=30, anomaly_count=30, max=0.000800, min=0.0.
    - cartservice-1 / trace.self.duration_p95 — global_P95=0.0, global_P5=0.0, points=30, anomaly_count=30, max=0.000400, min=0.0.
    - cartservice2-0 / trace.self.duration_p95 — global_P95=0.00011, global_P5=0.0, points=30, anomaly_count=30, max=0.000250, min=0.0.
    - cartservice-2 / trace.self.duration_mean — global_P95=0.0, global_P5=0.0, points=30, anomaly_count=30, max=0.000032, min=0.0.
    - Many adservice pods have trace.from_frontend-*.error_rate entries flagged (global_P95=0.0, global_P5=0.0, max=0.0, min=0.0) and were flagged for every point.
  - Most flagged trace names are latency/error metrics: trace.self.duration_p95, trace.self.duration_mean, trace.self.error_rate, and trace.from_frontend-*.error_rate.

- Key observation:
  - A large set of trace series had their global P95 and P5 equal to 0 (or very small), which led to every point in the 02:00–02:30 window being classified as anomalous under the rule (value >= global_P95 OR value <= global_P5). This explains why anomaly_count = points_in_window for many pairs.
  - Numeric values in the window are very small (milliseconds or zero) for the listed traces; many error_rate series are zero across the window.

If you want, next steps I can take:
- Show a concise list of pairs where global_P95 == global_P5 == 0 (to confirm threshold issue), or
- Inspect a few raw trace time series for selected cmdb_ids to verify the actual values across the window. Which do you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                         trace_name  global_P95  global_P5  points_in_window  anomaly_count      earliest_anomaly_ts  max_value_in_window  min_value_in_window
0    cartservice-0            trace.self.duration_p95     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000800                  0.0
1    cartservice-1            trace.self.duration_p95     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000400                  0.0
2   cartservice2-0            trace.self.duration_p95     0.00011        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000250                  0.0
3    cartservice-2           trace.self.duration_mean     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000032                  0.0
4      adservice-0   trace.from_frontend-0.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
5      adservice-0   trace.from_frontend-1.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
6      adservice-0   trace.from_frontend-2.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
7      adservice-1   trace.from_frontend-0.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
8      adservice-1   trace.from_frontend-1.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
9      adservice-2   trace.from_frontend-0.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
10     adservice-2   trace.from_frontend-1.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
11     adservice-2   trace.from_frontend-2.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
12    adservice2-0  trace.from_frontend2-0.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
13   cartservice-0   trace.from_frontend-0.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
14   cartservice-0   trace.from_frontend-1.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
15   cartservice-0   trace.from_frontend-2.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
16   cartservice-0              trace.self.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
17   cartservice-1   trace.from_frontend-0.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
18   cartservice-1   trace.from_frontend-1.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0
19   cartservice-1   trace.from_frontend-2.error_rate     0.00000        0.0              30.0           30.0  2022-03-21 02:00:00 CST             0.000000                  0.0, 1232)```
```

In [ ]:
# In[5]:

import re
import pandas as pd

# Reuse window_stats, merged, tz, fmt_min_max_ts from earlier kernel state.

# 1) Filter rows where kpi_name contains memory-related keywords (case-insensitive)
pattern = r'(memory|memory_failure|container_memory|GarbageCollector)'
mem_filtered = window_stats[window_stats['kpi_name'].str.contains(pattern, case=False, na=False)].copy()

# Ensure 'severity' exists
if 'severity' not in mem_filtered.columns:
    mem_filtered['severity'] = mem_filtered.apply(
        lambda r: max(abs(r['max_value_in_window'] - r['global_P95']), abs(r['min_value_in_window'] - r['global_P5'])), axis=1
    )

# 2) Top 50 sorted by anomaly_count desc then severity desc
top50_mem = (
    mem_filtered
    .sort_values(['anomaly_count','severity'], ascending=[False, False])
    .loc[:, ['cmdb_id','kpi_name','global_P95','global_P5','points_in_window','anomaly_count','earliest_anomaly_ts','max_value_in_window','min_value_in_window']]
    .head(50)
    .reset_index(drop=True)
)

# 3) Extract node identifiers from kpi_name (search for 'node-<N>')
mem_filtered['node'] = mem_filtered['kpi_name'].str.extract(r'(node-\d+)', flags=re.IGNORECASE, expand=False)
# Compute earliest anomaly unix per (cmdb_id,kpi_name) from merged (only where anomalies exist)
if 'merged' in globals():
    earliest_unix = (
        merged[merged['is_anom']==1]
        .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_anom_unix'})
    )
else:
    # Fallback: try to parse earliest_anomaly_ts strings back to unix (if merged not available)
    earliest_unix = mem_filtered[['cmdb_id','kpi_name']].copy()
    earliest_unix['earliest_anom_unix'] = pd.NaT

# Merge earliest_unix into mem_filtered
mem_filtered = mem_filtered.merge(earliest_unix, on=['cmdb_id','kpi_name'], how='left')

# Node summary: node -> count of distinct cmdb_id affected and earliest anomaly ts (min)
node_grp = (
    mem_filtered[mem_filtered['node'].notna()]
    .groupby('node', as_index=False)
    .agg(
        count_distinct_cmdb_ids = ('cmdb_id', lambda s: int(s.nunique())),
        earliest_anom_unix = ('earliest_anom_unix', 'min')
    )
)
# Format earliest_anom_unix to UTC+8 strings
node_grp['earliest_anomaly_ts'] = node_grp['earliest_anom_unix'].apply(lambda x: fmt_min_max_ts(x) if pd.notna(x) else None)
node_summary = node_grp[['node','count_distinct_cmdb_ids','earliest_anomaly_ts']].sort_values(['count_distinct_cmdb_ids','earliest_anomaly_ts'], ascending=[False, True]).reset_index(drop=True)

# 4) Total number of distinct cmdb_id affected in filtered set
total_distinct_cmdbids = int(mem_filtered['cmdb_id'].nunique())

# Overall earliest anomaly timestamp across memory-related KPIs (min earliest_anom_unix)
overall_earliest_unix = mem_filtered['earliest_anom_unix'].min()
overall_earliest_anomaly_ts = fmt_min_max_ts(overall_earliest_unix) if pd.notna(overall_earliest_unix) else None

# Compact outputs
top50_mem, node_summary, total_distinct_cmdbids, overall_earliest_anomaly_ts

```
Out[5]:
```
Summary (memory-related KPI scan, window 2022-03-21 02:00–02:30 UTC+8):

- Filtered set: memory-related KPIs (kpi_name contains memory / memory_failure / container_memory / GarbageCollector).
- Top 50 matching KPI rows (sorted by anomaly_count desc then severity) all show sustained anomalies across the window (most rows have points_in_window = 31 and anomaly_count = 31; earliest_anomaly_ts = 2022-03-21 02:00:00 UTC+8). Example top entries:
  - recommendationservice-2 / container.node-6.container_memory_rss — global_P95 39,837,696; global_P5 39,718,912; points 31; anomaly_count 31; max 48,285,696; min 39,718,912.
  - recommendationservice-2 / container.node-6.container_memory_mapped_file — global_P95 675,840; global_P5 405,504; points 31; anomaly_count 31; max 3,244,032; min 405,504.
  - adservice-1 / container.node-6.container_memory_rss — global_P95 139,051,008; global_P5 138,915,840; points 31; anomaly_count 31; max/min ≈ 139,051,008.
  - Several runtime GarbageCollector KPIs for adservice also appear in the top 50.
- Node-level summary (extracted from kpi_name):
  - node-6: 34 distinct cmdb_id affected; earliest anomaly 2022-03-21 02:00:00 UTC+8.
  - node-5: 6 distinct cmdb_id affected; earliest anomaly 2022-03-21 02:00:00 UTC+8.
- Totals:
  - Total distinct cmdb_id affected in this filtered memory-related set: 42.
  - Overall earliest anomaly timestamp across these memory-related KPIs: 2022-03-21 02:00:00 UTC+8.

Interpretation: memory-related KPIs (notably container memory RSS/mapped_file and some GC metrics) show widespread, sustained deviations beginning at the start of the 02:00 window, concentrated primarily on node-6 (and to a lesser extent node-5).

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name   global_P95    global_P5  points_in_window  anomaly_count      earliest_anomaly_ts  max_value_in_window  min_value_in_window
0   recommendationservice-2              container.node-6.container_memory_rss   39837696.0   39718912.0              31.0           31.0  2022-03-21 02:00:00 CST           48285696.0           39718912.0
1   recommendationservice-2      container.node-6.container_memory_mapped_file     675840.0     405504.0              31.0           31.0  2022-03-21 02:00:00 CST            3244032.0             405504.0
2          paymentservice-2              container.node-6.container_memory_rss   34316288.0   33861632.0              31.0           31.0  2022-03-21 02:00:00 CST           33861632.0           33861632.0
3               adservice-1              container.node-6.container_memory_rss  139051008.0  138915840.0              31.0           31.0  2022-03-21 02:00:00 CST          139051008.0          139051008.0
4         shippingservice-2              container.node-5.container_memory_rss   13078528.0   13062144.0              31.0           31.0  2022-03-21 02:00:00 CST           13078528.0           12926976.0
..                      ...                                                ...          ...          ...               ...            ...                      ...                  ...                  ...
45                adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...    3407872.0    3407872.0              31.0           31.0  2022-03-21 02:00:00 CST            3407872.0            3407872.0
46                adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...    3407872.0    3407872.0              31.0           31.0  2022-03-21 02:00:00 CST            3407872.0            3407872.0
47                adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...   68886528.0   68886528.0              31.0           31.0  2022-03-21 02:00:00 CST           68886528.0           68886528.0
48                adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...   68886528.0   68886528.0              31.0           31.0  2022-03-21 02:00:00 CST           68886528.0           68886528.0
49                adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...    2555904.0    2555904.0              31.0           31.0  2022-03-21 02:00:00 CST            2555904.0            2555904.0

[50 rows x 9 columns],      node  count_distinct_cmdb_ids      earliest_anomaly_ts
0  node-6                       34  2022-03-21 02:00:00 CST
1  node-5                        6  2022-03-21 02:00:00 CST, 42, '2022-03-21 02:00:00 CST')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import re
import pandas as pd
import pytz

# Reuse variables from kernel: metric_df, start_ts, end_ts, tz, fmt_min_max_ts if available
tz = pytz.timezone('Asia/Shanghai')
# If start_ts/end_ts not defined (defensive), define them for 2022-03-21 02:00..02:30 UTC+8
try:
    start_ts, end_ts
except NameError:
    start_ts = int(pd.Timestamp('2022-03-21 02:00:00', tz=tz).timestamp())
    end_ts   = int(pd.Timestamp('2022-03-21 02:30:00', tz=tz).timestamp())

# 1) Filter rows matching memory_fail patterns, value>0, and within time window
pattern = r'(memory_fail|memory\.fail|memory_failures)'
memfail_rows = metric_df[
    metric_df['kpi_name'].str.contains(pattern, case=False, na=False)
    & (metric_df['value'] > 0)
    & (metric_df['timestamp'] >= start_ts)
    & (metric_df['timestamp'] <= end_ts)
].copy()

# 2) For each distinct cmdb_id compute earliest timestamp with value>0 (min), count, max value in window
# Also attempt to extract node id from kpi_name (node-<N>); collect unique nodes per cmdb_id and join if multiple
memfail_rows['node_extracted'] = memfail_rows['kpi_name'].str.extract(r'(node-\d+)', flags=re.IGNORECASE, expand=False)

grp = memfail_rows.groupby('cmdb_id', as_index=False).agg(
    earliest_ts_unix = ('timestamp', 'min'),
    count_points_gt0 = ('value', 'count'),
    max_value_in_window = ('value', 'max'),
    nodes = ('node_extracted', lambda s: ','.join(sorted({x for x in s.dropna()})))
)

# Format earliest timestamp to UTC+8 string
grp['earliest_timestamp_with_value_gt0'] = grp['earliest_ts_unix'].apply(lambda x: fmt_min_max_ts(x) if pd.notna(x) else None)

# Select and order columns
mem_fail_results = grp[[
    'cmdb_id',
    'nodes',
    'earliest_timestamp_with_value_gt0',
    'count_points_gt0',
    'max_value_in_window'
]].rename(columns={
    'nodes': 'node',
    'count_points_gt0': 'count_of_points_with_value_gt0'
})

# 3) Sort top 50 by count desc then earliest asc
mem_fail_top50 = mem_fail_results.sort_values(
    ['count_of_points_with_value_gt0','earliest_timestamp_with_value_gt0'],
    ascending=[False, True]
).head(50).reset_index(drop=True)

# 4) Node-level summary: for each node found, number_of_distinct_cmdb_ids_with_memory_failures and node-level earliest timestamp
# Use the per-cmdb_id data to avoid double-counting
nodes_expanded = mem_fail_top50[['cmdb_id','node','earliest_timestamp_with_value_gt0']].copy()
# Drop rows without node
nodes_expanded = nodes_expanded[nodes_expanded['node'].notna() & (nodes_expanded['node'] != '')]

if not nodes_expanded.empty:
    node_summary = (
        nodes_expanded
        .groupby('node', as_index=False)
        .agg(
            number_of_distinct_cmdb_ids_with_memory_failures = ('cmdb_id', lambda s: int(s.nunique())),
            node_earliest_timestamp_unformatted = ('earliest_timestamp_with_value_gt0', 'min')
        )
    )
    node_summary = node_summary.rename(columns={'node_earliest_timestamp_unformatted':'node_earliest_timestamp'})
else:
    node_summary = pd.DataFrame(columns=['node','number_of_distinct_cmdb_ids_with_memory_failures','node_earliest_timestamp'])

# Overall counts and earliest
total_distinct_cmdbids_mem_fail = int(mem_fail_results['cmdb_id'].nunique())
overall_earliest_unix = grp['earliest_ts_unix'].min() if not grp.empty else None
overall_earliest_anomaly_ts = fmt_min_max_ts(overall_earliest_unix) if pd.notna(overall_earliest_unix) else None

# Compact outputs
mem_fail_top50, node_summary, total_distinct_cmdbids_mem_fail, overall_earliest_anomaly_ts

```
Out[6]:
```
summary = (
    "Memory-failure findings (window 2022-03-21 02:00–02:30 UTC+8):\n"
    "- Total distinct cmdb_id with memory-failure metrics > 0: 27.\n"
    "- Node summary: node-6 affected 21 distinct cmdb_ids (earliest anomaly 2022-03-21 02:00:00 UTC+8); "
    "node-5 affected 6 distinct cmdb_ids (earliest anomaly 2022-03-21 02:00:00 UTC+8).\n"
    "- Top affected cmdb_ids (by count_of_points_with_value_gt0):\n"
    "  * currencyservice-0 — 93 points >0; max value 21202.5; earliest 2022-03-21 02:00:00\n"
    "  * currencyservice-2 — 93 points >0; max value 19074.0; earliest 2022-03-21 02:00:00\n"
    "  * currencyservice2-0 — 93 points >0; max value 19915.5; earliest 2022-03-21 02:00:00\n"
    "  * paymentservice-0 — 69 points >0; max value 33.0; earliest 2022-03-21 02:00:00\n"
    "  * adservice-0 — 62 points >0; max value 132.0; earliest 2022-03-21 02:00:00\n"
    "  * (other entries include shippingservice-2, recommendationservice-*, etc., with smaller counts down to 2–4 points).\n"
    "- Overall earliest memory-failure timestamp observed in this filtered set: 2022-03-21 02:00:00 UTC+8.\n"
    "Interpretation: memory-failure events (value>0) are concentrated on node-6 (majority of affected cmdb_ids) with node-5 also impacted; several services (notably currencyservice pods) show the highest counts of failure points during the window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id    node earliest_timestamp_with_value_gt0  count_of_points_with_value_gt0  max_value_in_window
0          currencyservice-0  node-6           2022-03-21 02:00:00 CST                              93              21202.5
1          currencyservice-2  node-6           2022-03-21 02:00:00 CST                              93              19074.0
2         currencyservice2-0  node-6           2022-03-21 02:00:00 CST                              93              19915.5
3           paymentservice-0  node-6           2022-03-21 02:00:00 CST                              69                 33.0
4                adservice-0  node-6           2022-03-21 02:00:00 CST                              62                132.0
..                       ...     ...                               ...                             ...                  ...
22         shippingservice-2  node-5           2022-03-21 02:17:00 CST                               6                 82.5
23   recommendationservice-0  node-6           2022-03-21 02:26:00 CST                               4               4504.5
24   recommendationservice-2  node-6           2022-03-21 02:26:00 CST                               4               9058.5
25  recommendationservice2-0  node-6           2022-03-21 02:26:00 CST                               4               5659.5
26   recommendationservice-1  node-6           2022-03-21 02:27:00 CST                               2              10098.0

[27 rows x 5 columns],      node  number_of_distinct_cmdb_ids_with_memory_failures  node_earliest_timestamp
0  node-5                                                 6  2022-03-21 02:00:00 CST
1  node-6                                                21  2022-03-21 02:00:00 CST, 27, '2022-03-21 02:00:00 CST')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import re
import pandas as pd
import pytz

# Reuse existing variables where possible: metric_df, start_ts, end_ts, fmt_min_max_ts, tz
# Defensive definitions if not present
tz = pytz.timezone('Asia/Shanghai')
try:
    start_ts, end_ts, metric_df, fmt_min_max_ts
except NameError:
    start_ts = int(pd.Timestamp('2022-03-21 02:00:00', tz=tz).timestamp())
    end_ts   = int(pd.Timestamp('2022-03-21 02:30:00', tz=tz).timestamp())
    metric_df = pd.read_csv('metric.csv')
    # helper to format epoch seconds to UTC+8 formatted strings
    def fmt_min_max_ts(s):
        if s is None or (isinstance(s, float) and pd.isna(s)):
            return None
        dt = pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz)
        return dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Filter metric rows: kpi_name contains memory_fail|memory.fail|memory_failures (case-insensitive), value>0, within window
pattern = r'(memory_fail|memory\.fail|memory_failures)'
mask = metric_df['kpi_name'].str.contains(pattern, case=False, na=False)
memfail_rows = metric_df[mask & (metric_df['value'] > 0) & (metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Extract node id from kpi_name if present
memfail_rows['node'] = memfail_rows['kpi_name'].str.extract(r'(node-\d+)', flags=re.IGNORECASE, expand=False)

# Group by node and compute required aggregates
rows = []
for node, grp in memfail_rows[memfail_rows['node'].notna()].groupby('node'):
    node_earliest_unix = int(grp['timestamp'].min())
    # cmdb_ids that had event at that earliest timestamp (unique, preserve appearance order)
    cmdbs_at_earliest = grp[grp['timestamp'] == node_earliest_unix]['cmdb_id'].dropna().astype(str)
    # preserve original order and unique
    seen = []
    for v in cmdbs_at_earliest:
        if v not in seen:
            seen.append(v)
    cmdbs_at_earliest_limited = seen[:10]
    total_distinct_cmdb_ids = int(grp['cmdb_id'].nunique())
    rows.append({
        'node_id': node,
        'node_earliest_memory_failure_ts': fmt_min_max_ts(node_earliest_unix),
        'cmdb_ids_at_node_earliest_ts': ','.join(cmdbs_at_earliest_limited),
        'total_distinct_cmdb_ids_with_memory_failures_on_node': total_distinct_cmdb_ids
    })

node_summary_df = pd.DataFrame(rows).sort_values(['total_distinct_cmdb_ids_with_memory_failures_on_node','node_id'], ascending=[False, True]).reset_index(drop=True)

# Display compact node-level summary (nodes found)
node_summary_df

```
Out[7]:
```
Summary — node-level memory-failure events (window 2022-03-21 02:00–02:30 UTC+8):

- node-6
  - Earliest memory-failure timestamp: 2022-03-21 02:00:00 (UTC+8)
  - Example cmdb_ids observed at that earliest timestamp (first up to 10): adservice-0, adservice-1, adservice2-0, cartservice-0, ... (list limited to first 10)
  - Total distinct cmdb_ids with memory-failures on node-6: 21

- node-5
  - Earliest memory-failure timestamp: 2022-03-21 02:00:00 (UTC+8)
  - cmdb_ids observed at that earliest timestamp: cartservice2-0
  - Total distinct cmdb_ids with memory-failures on node-5: 6

Interpretation: memory-failure events are concentrated on node-6 (majority of affected pods), with node-5 also impacted; both nodes show earliest events at the start of the 02:00 window.

The original code execution output of IPython Kernel is also provided below for reference:

node_id node_earliest_memory_failure_ts                       cmdb_ids_at_node_earliest_ts  total_distinct_cmdb_ids_with_memory_failures_on_node
0  node-6         2022-03-21 02:00:00 CST  adservice-0,adservice-1,adservice2-0,cartservi...                                                 21   
1  node-5         2022-03-21 02:00:00 CST                                     cartservice2-0                                                  6```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse existing kernel variables where possible:
# log_df, mem_fail_results (contains cmdb_ids with memory-failures), start_ts, end_ts, fmt_min_max_ts, tz
tz = pytz.timezone('Asia/Shanghai')

# Defensive fallback if mem_fail_results not present
try:
    target_cmdbs = mem_fail_results['cmdb_id'].unique().tolist()
except NameError:
    # fallback to mem_fail_top50 if available
    try:
        target_cmdbs = mem_fail_top50['cmdb_id'].unique().tolist()
    except NameError:
        target_cmdbs = []

# Ensure start_ts/end_ts defined (UTC+8)
try:
    start_ts, end_ts
except NameError:
    start_ts = int(pd.Timestamp('2022-03-21 02:00:00', tz=tz).timestamp())
    end_ts   = int(pd.Timestamp('2022-03-21 02:30:00', tz=tz).timestamp())

# Ensure fmt_min_max_ts exists
def _fmt_ts(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    return pd.to_datetime(int(s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

try:
    fmt_min_max_ts
except NameError:
    fmt_min_max_ts = _fmt_ts

# Filter log rows by time window and target cmdb_ids
logs_window = log_df[
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts) &
    (log_df['cmdb_id'].isin(target_cmdbs))
].copy()

# Total log rows per cmdb_id
total_rows = logs_window.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'total_log_rows_in_window'})

# log.error_count stats
logs_error = logs_window[logs_window['log_name']=='log.error_count'].copy()
if not logs_error.empty:
    error_stats = logs_error.groupby('cmdb_id').agg(
        max_log_error_count=('value','max'),
        sum_log_error_count=('value','sum'),
        earliest_ts_error_gt0_unix=('timestamp', lambda s: int(s[logs_error.loc[s.index,'value']>0].min()) if any(logs_error.loc[s.index,'value']>0) else pd.NA)
    ).reset_index()
else:
    error_stats = pd.DataFrame(columns=['cmdb_id','max_log_error_count','sum_log_error_count','earliest_ts_error_gt0_unix'])

# log.row_count stats
logs_row = logs_window[logs_window['log_name']=='log.row_count'].copy()
if not logs_row.empty:
    row_stats = logs_row.groupby('cmdb_id', as_index=False).agg(max_log_row_count=('value','max'))
else:
    row_stats = pd.DataFrame(columns=['cmdb_id','max_log_row_count'])

# Merge summaries
summary = total_rows.merge(error_stats, on='cmdb_id', how='left').merge(row_stats, on='cmdb_id', how='left')

# Format earliest timestamp where error_count > 0
if 'earliest_ts_error_gt0_unix' in summary.columns:
    summary['earliest_ts_where_log_error_count_gt0'] = summary['earliest_ts_error_gt0_unix'].apply(lambda x: fmt_min_max_ts(x) if pd.notna(x) else None)
    summary = summary.drop(columns=['earliest_ts_error_gt0_unix'])
else:
    summary['earliest_ts_where_log_error_count_gt0'] = None

# Replace NaNs for max/sum with None for compactness
for c in ['max_log_error_count','sum_log_error_count','max_log_row_count']:
    if c in summary.columns:
        summary[c] = summary[c].where(pd.notna(summary[c]), None)

# Top 20 cmdb_ids sorted by max_log_error_count desc (None treated as -inf), then total_log_rows_in_window desc
summary['max_log_error_count_sort'] = summary['max_log_error_count'].apply(lambda x: x if x is not None else -1)
top20_logs_summary = summary.sort_values(['max_log_error_count_sort','total_log_rows_in_window'], ascending=[False, False])\
    .drop(columns=['max_log_error_count_sort'])\
    .head(20).reset_index(drop=True)

# Sample raw log rows where log.error_count > 0 (limit to first 10 matches), format timestamp to UTC+8
sample_error_rows = logs_error[logs_error['value']>0].head(10).copy()
if not sample_error_rows.empty:
    sample_error_rows['ts_utc8'] = pd.to_datetime(sample_error_rows['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    sample_error_rows = sample_error_rows[['ts_utc8','cmdb_id','log_name','value']].reset_index(drop=True)

# Final compact outputs
top20_logs_summary, sample_error_rows

```
Out[8]:
```
Summary (logs for the 27 cmdb_ids with memory-failures, window 2022-03-21 02:00–02:30 UTC+8):

- No nonzero error log events found:
  - Across all filtered cmdb_ids, log.error_count was zero for every matching row in the window (max_log_error_count = 0 and sum_log_error_count = 0 for all).
  - Consequently, there are no timestamps where log.error_count > 0 (earliest_ts_where_log_error_count_gt0 = None for all).
  - Sample raw log rows with log.error_count > 0: none (no matches).

- Per-cmdb_id summary (top 20 by max_log_error_count then total rows) — examples:
  - adservice-0: total_log_rows_in_window = 60, max_log_row_count = 56
  - adservice-1: total_log_rows_in_window = 60, max_log_row_count = 46
  - cartservice-0: total_log_rows_in_window = 60, max_log_row_count = 328
  - currencyservice-0: total_log_rows_in_window = 60, max_log_row_count = 290
  - frontend-1: total_log_rows_in_window = 60, max_log_row_count = 728
  - recommendationservice-* and shippingservice-2, emailservice-*, paymentservice-* also appear with 50–60 rows and their respective max row_count values.

- Interpretation: During the 02:00–02:30 window, memory-failure–affected pods produced normal log.error_count (zero) values; no error_count spikes were observed. Logs do show varying log.row_count volumes per pod, but no log.error_count events to correlate with the memory-failures.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  total_log_rows_in_window  max_log_error_count  sum_log_error_count  max_log_row_count earliest_ts_where_log_error_count_gt0
0               adservice-0                        60                  0.0                  0.0               56.0                                  None
1               adservice-1                        60                  0.0                  0.0               46.0                                  None
2               adservice-2                        60                  0.0                  0.0               44.0                                  None
3             cartservice-0                        60                  0.0                  0.0              328.0                                  None
4             cartservice-1                        60                  0.0                  0.0              320.0                                  None
5             cartservice-2                        60                  0.0                  0.0              329.0                                  None
6         currencyservice-0                        60                  0.0                  0.0              290.0                                  None
7         currencyservice-1                        60                  0.0                  0.0              293.0                                  None
8         currencyservice-2                        60                  0.0                  0.0              302.0                                  None
9                frontend-1                        60                  0.0                  0.0              728.0                                  None
10               frontend-2                        60                  0.0                  0.0              304.0                                  None
11  recommendationservice-0                        60                  0.0                  0.0               83.0                                  None
12  recommendationservice-1                        60                  0.0                  0.0               84.0                                  None
13  recommendationservice-2                        60                  0.0                  0.0               84.0                                  None
14        shippingservice-2                        60                  0.0                  0.0               46.0                                  None
15           emailservice-2                        54                  0.0                  0.0                9.0                                  None
16         paymentservice-1                        54                  0.0                  0.0                8.0                                  None
17         paymentservice-2                        54                  0.0                  0.0                8.0                                  None
18           emailservice-1                        52                  0.0                  0.0                6.0                                  None
19           emailservice-0                        50                  0.0                  0.0                8.0                                  None, Empty DataFrame
Columns: [timestamp, cmdb_id, log_name, value]
Index: [])```
```